<a href="https://colab.research.google.com/github/NathanMcCormack/Principles_of_ML_Assignment2/blob/main/PML_Assignment2_26244470.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Principles of Machine Learning - Assignment 2
##Nathan McCormack
##ID: 26244470

In [1]:
import pandas as pd
from google.colab import files

In [2]:
uploaded = files.upload()
df = pd.read_csv("Heart_failure_data.csv")

Saving Heart_failure_data.csv to Heart_failure_data.csv


##Analysing Dataset

In [3]:
print("Dataset shape:", df.shape) #prints rows & cols of dataset
display(df.head()) #shows us the first 5 patients
df.info() #shows us column names and non null counts so we can see incomplete data

#based off the non null vallues shown, we can see there are a lot of missing values in this dataset, next I'll look exactly where they lie

Dataset shape: (299, 13)


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,patient_id
0,75.0,0.0,582.0,0.0,20.0,1.0,265000.00,1.9,130.0,1.0,0.0,4.0,0
1,55.0,0.0,7861.0,0.0,38.0,0.0,263358.03,1.1,136.0,1.0,0.0,6.0,1
2,65.0,NaN,146.0,0.0,20.0,0.0,162000.00,1.3,129.0,1.0,NaN,7.0,2
3,50.0,1.0,111.0,0.0,20.0,0.0,210000.00,NaN,137.0,1.0,0.0,7.0,3
4,65.0,1.0,160.0,1.0,20.0,0.0,327000.00,2.7,116.0,0.0,0.0,8.0,4


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       270 non-null    float64
 1   anaemia                   270 non-null    float64
 2   creatinine_phosphokinase  270 non-null    float64
 3   diabetes                  270 non-null    float64
 4   ejection_fraction         270 non-null    float64
 5   high_blood_pressure       270 non-null    float64
 6   platelets                 270 non-null    float64
 7   serum_creatinine          270 non-null    float64
 8   serum_sodium              270 non-null    float64
 9   sex                       270 non-null    float64
 10  smoking                   270 non-null    float64
 11  time                      270 non-null    float64
 12  patient_id                299 non-null    int64  
dtypes: float64(12), int64(1)
memory usage: 30.5 KB


In [4]:
print(df.isnull().sum()) #checks the location and count of each missing value.
#we can see each feature has 29 missing values, this is likley going to be too much to ignore as we have a smaller dataset, so I will likely
#have to fill in these values using a trusted prediction algorithm.
#Thankfuly all patient IDs are present, so we know at least that each row corresponds to a single patient.

age                         29
anaemia                     29
creatinine_phosphokinase    29
diabetes                    29
ejection_fraction           29
high_blood_pressure         29
platelets                   29
serum_creatinine            29
serum_sodium                29
sex                         29
smoking                     29
time                        29
patient_id                   0
dtype: int64


In [6]:
#isnull checks every cell and returns true,any(axis=1) checks each row to see whether at least one value is missing
#sum then counts tehse true values
print("Patients with missing values:", df.isnull().any(axis=1).sum())

Patients with missing values: 211


In [9]:
#here im checking for any patients who have complete records. These will be especially important when filling in the missing data later on
print("Patients with missing data:", df.dropna().shape[0])

Patients with missing data: 88


After a quick analysis of these missing values, I am going to look for ways to imputate new data. I could potentially get a mean of a missing value from other patients who have similar binary attributes (age, smoking, diabetes). If using a mean I will have to be cautious of outliers that may heavily affect the output. Due to this a median may be preffered.

In [10]:
df.describe().T #.T transposes the data making it easier for us to read.

,count,mean,std,min,25%,50%,75%,max
age,270.0,60.934570,11.838766,40.0,52.00,60.0,70.0,95.0
anaemia,270.0,0.437037,0.496941,0.0,0.00,0.0,1.0,1.0
creatinine_phosphokinase,270.0,574.266667,958.532760,23.0,119.50,250.0,582.0,7861.0
diabetes,270.0,0.411111,0.492949,0.0,0.00,0.0,1.0,1.0
ejection_fraction,270.0,37.622222,11.715885,14.0,30.00,38.0,45.0,70.0
high_blood_pressure,270.0,0.355556,0.479570,0.0,0.00,0.0,1.0,1.0
platelets,270.0,261522.410185,97633.259068,25100.0,210250.00,261000.0,302000.0,850000.0
serum_creatinine,270.0,1.355519,0.981258,0.5,0.90,1.1,1.4,9.4
serum_sodium,270.0,136.722222,4.226033,113.0,134.00,137.0,139.0,148.0
sex,270.0,0.640741,0.480674,0.0,0.00,1.0,1.0,1.0


from the table above we can immedietely see some issues with the dataset:
- creatinine_phosphokinase: min = 23, mean = 574, max = 7861
- platelets: min = 25,100, mean = 261,522, max = 850,000

for platelets the mean and median are almost identical, whereas for creatine phosphokinase the mean is nearly twice the median, which tells us its right side biased.

In [11]:
features = df.drop(columns=["patient_id", "time"]) #using drop to remove non feature variables

correlations = features.corr() #using .corr to get the correlations, defaults using Pearson Method

display(correlations.round(2)) #displays corr values to 2 decimal places

#Correlation scores from -1 to 1, 1 being the strongest correlation
#looking at these results there are no correlations that are strong enough to base new values off. Our strongest correlation was smoking and sex
#at 0.46. and our strongest negative correlation was -0.24 between creatine phospho and anaemia, which is still far too weak.

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking
age,1.00,0.08,-0.01,-0.05,0.03,0.07,0.03,0.12,-0.06,0.07,0.02
anaemia,0.08,1.00,-0.24,-0.03,0.04,0.03,-0.01,0.08,0.05,-0.14,-0.14
creatinine_phosphokinase,-0.01,-0.24,1.00,-0.02,-0.02,-0.04,0.03,-0.08,0.01,0.08,-0.02
diabetes,-0.05,-0.03,-0.02,1.00,-0.01,-0.00,0.12,-0.05,-0.08,-0.13,-0.08
ejection_fraction,0.03,0.04,-0.02,-0.01,1.00,0.04,0.10,-0.05,0.16,-0.13,-0.09
high_blood_pressure,0.07,0.03,-0.04,-0.00,0.04,1.00,0.08,-0.01,0.11,-0.09,-0.05
platelets,0.03,-0.01,0.03,0.12,0.10,0.08,1.00,-0.03,0.05,-0.11,0.07
serum_creatinine,0.12,0.08,-0.08,-0.05,-0.05,-0.01,-0.03,1.00,-0.23,0.03,-0.00
serum_sodium,-0.06,0.05,0.01,-0.08,0.16,0.11,0.05,-0.23,1.00,-0.03,0.01
sex,0.07,-0.14,0.08,-0.13,-0.13,-0.09,-0.11,0.03,-0.03,1.00,0.46


After running these tests I think I will take the approach of using a median for all the non binary feature variables, and use a mode for all the binary feature variables.

Using a median will remove the bias of outliers.
Using a mode may actually be bias towards a heavy majority, which could affect our results  

##Data Preprocessing

In [12]:
#creating a copy of the dataset
df_clean = df.copy()

In [13]:
#removing the non feature variables
df_clean = df_clean.drop(columns=["patient_id", "time"])

In [14]:
nonbinary_cols = ["age", "creatinine_phosphokinase", "ejection_fraction", "platelets", "serum_creatinine", "serum_sodium"]

In [15]:
binary_cols = ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking"]